Research Question 6: Pipeline

To note at the beginning is the data was adjusted multiple times to fit the scope of the question: 
- we wanted to include 5 years of data, 2021 to 2025, but we noticed that much of the data was very patchy pre-2023, so we settled for 2023 to 2025 instead
- some countries had incomplete data, so we decided to exclude them as well, even though they are technically part of the relevant regions; United Kingdom, Albania, Malta, Iceland, Cyprus
- some countries had misleading reporting, for instance, Ireland's reported total capacity for 2023 did not match the absolute lowest value one could gather from the GEM trackers, as well as missing the values for 2024 and 2025, which is why another step was added to validate, and if necessary add values

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
import os
import glob
import warnings
import sys

Manual Data Processing

Before any of the coding, data of all potentially relevant facilities, which was provided by the GEM trackers, were filtered, sorted and output in separate files according to the energy source. Only facilities with the status "operating" or "retired" were included.
- Output: Independent files for Solar, Wind, Hydro, and Bioenergy.
- Format: [Country, Capacity (MW), Status, Start Year, Retired Year, Latitude, Longitude]

Spatial Clustering

This is the first program, which reads the aforementioned manually prepared files
First, for each year, it identifies the relevant facilities. Facilities that were opened or retired that year are still counted towards it.
Every facilites is assigned a fixed 111 x 111km plot, and the total capacity is clustered. 
- Output: European_Renewable_Cluster.csv
- Format: [Region, Technology, Country, Year, Latitude_Round, Longitude_Round, Total_Capacity]

In [ ]:
#AI assisted code.
warnings.filterwarnings('ignore')

INPUT_FILES = {
    'Bioenergy': 'European_Bioenergy_Data.xlsx',
    'Hydro': 'European_Hydro_Data.xlsx',
    'Solar': 'European_Solar_Data.xlsx',
    'Wind': 'European_Wind_Data.xlsx' 
}

NORTHERN_EUROPE = ['Denmark', 'Sweden', 'Finland', 'Lithuania', 'Ireland', 'Estonia', 'Latvia', 'Norway']
SOUTHERN_EUROPE = ['Portugal', 'Italy', 'Spain', 'Serbia', 'Croatia', 'Bosnia and Herzegovina', 'Greece', 'Montenegro', 'Slovenia', 'North Macedonia']

TARGET_YEARS = [2023, 2024, 2025]

def get_region(country):
    if country in NORTHERN_EUROPE: return 'Northern Europe'
    if country in SOUTHERN_EUROPE: return 'Southern Europe'
    return None

def process_trackers():
    all_data = []
    
    print("--- Starting Yearly Equal-Area Clustering ---")
    for tech, filename in INPUT_FILES.items():
        if not os.path.exists(filename):
            print(f"[Skipping] {filename} not found.")
            continue
            
        print(f"Processing {tech}...")
        df = pd.read_excel(filename)
        
        # --- EQUAL-AREA COSINE CORRECTION ---
        df['Latitude_Round'] = df['Latitude'].round(0)
        lat_radians = np.radians(df['Latitude_Round'])
        lon_step = 1.0 / np.clip(np.cos(lat_radians), 0.01, 1.0)
        df['Longitude_Round'] = (df['Longitude'] / lon_step).round(0) * lon_step
        df['Longitude_Round'] = df['Longitude_Round'].round(2)
        
        # --- YEARLY FILTERING ---
        for year in TARGET_YEARS:
            # Condition: Built before/during the target year, and hasn't retired yet
            started = df['Start Year'].isna() | (df['Start Year'] <= year)
            not_retired = df['Retired Year'].isna() | (df['Retired Year'] >= year)
            
            active_df = df[started & not_retired].copy()
            if active_df.empty:
                continue
                
            active_df['Technology'] = tech
            active_df['Year'] = year
            all_data.append(active_df)

    if not all_data:
        print("No valid data found.")
        return pd.DataFrame()

    master_df = pd.concat(all_data, ignore_index=True)
    
    # --- IMPLICIT SPLIT & AGGREGATION ---
    # Grouping directly by Country alongside coordinates ensures border-spanning 
    # capacities separate cleanly into independent national records.
    clusters = master_df.groupby(['Technology', 'Country', 'Year', 'Latitude_Round', 'Longitude_Round']).agg(
        Total_Capacity=('Capacity (MW)', 'sum')
    ).reset_index()
    
    clusters['Region'] = clusters['Country'].apply(get_region)
    clusters = clusters.dropna(subset=['Region']) 
    
    # Enforce strict formatting and column order
    clusters = clusters[['Region', 'Technology', 'Country', 'Year', 'Latitude_Round', 'Longitude_Round', 'Total_Capacity']]
    clusters['Total_Capacity'] = clusters['Total_Capacity'].round(1)
    
    # Sort for readability (Largest capacities at top)
    clusters = clusters.sort_values(by=['Region', 'Technology', 'Country', 'Year', 'Total_Capacity'], ascending=[True, True, True, True, False])
    
    return clusters

if __name__ == "__main__":
    final_clusters = process_trackers()
    
    if not final_clusters.empty:
        csv_out = "European_Renewable_Clusters.csv"
        final_clusters.to_csv(csv_out, index=False)
        print(f"\n✓ Master CSV saved to: {csv_out}")
            
        print("*** ALL EXPORTS COMPLETE ***")

Weather Data Fetching

This program fetches the data for 7 unique coordinates for each cluster, for each year, from Open-Meteo API.
Hourly variables are aggregated into daily values, in this instance the mean is calculated
Auto-timezones were fetched instead of UTC to align local solar and wind cycles.
To minimize API calls, unique clusters are identified first.

- Output: European_Weather_Data_2023_2025.csv
- Format: [Lat_Rounded, Lon_Rounded, time, shortwave_radiation_sum, temperature_2m_max, wind_gusts_10m_max, precipitation_sum, apparent_temperature_min, wind_speed_100m, wind_speed_100m_cubed, snow_depth]


In [ ]:
#AI assisted code.
warnings.filterwarnings('ignore')

INPUT_FILE = "European_Renewable_Clusters.csv"
OUTPUT_FILE = "European_Weather_Data_2023_2025.csv"
TARGET_YEARS = [2023, 2024, 2025]

DAILY_VARS = [
    "shortwave_radiation_sum", 
    "temperature_2m_max", 
    "wind_gusts_10m_max", 
    "precipitation_sum", 
    "apparent_temperature_min"
]

HOURLY_VARS = [
    "wind_speed_100m", 
    "snow_depth"
]

def load_coordinates():
    print(f"Reading {INPUT_FILE}...")
    df = pd.read_csv(INPUT_FILE)
    
    # We drop the political geography entirely. 
    # We only care: Did a grid square contain ANY active infrastructure in a specific year?
    unique_coords = df[['Year', 'Latitude_Round', 'Longitude_Round']].drop_duplicates()
    return unique_coords

def get_completed_coordinates():
    if not os.path.exists(OUTPUT_FILE):
        return set()
    
    try:
        df = pd.read_csv(OUTPUT_FILE, usecols=['time', 'Lat_Rounded', 'Lon_Rounded'])
        df['Fetched_Year'] = df['time'].str[:4].astype(int)
        
        completed_df = df[['Fetched_Year', 'Lat_Rounded', 'Lon_Rounded']].drop_duplicates()
        
        completed = set(
            (int(year), round(lat, 2), round(lon, 2)) 
            for year, lat, lon in zip(
                completed_df['Fetched_Year'], 
                completed_df['Lat_Rounded'], 
                completed_df['Lon_Rounded']
            )
        )
        return completed
    except Exception as e:
        print(f"Could not read existing progress: {e}")
        return set()

def fetch_and_aggregate_weather(row):
    year = int(row['Year'])
    lat, lon = row['Latitude_Round'], row['Longitude_Round']
    
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": ",".join(DAILY_VARS),
        "hourly": ",".join(HOURLY_VARS),
        "timezone": "auto" 
    }
    
    max_retries = 3
    for attempt in range(max_retries):
        response = requests.get(url, params=params)
        
        if response.status_code == 429:
            if attempt == 0:
                print(f"  [Rate Limited] Minutely limit hit. Pausing for 65s...")
                time.sleep(65)
                continue
            elif attempt == 1:
                print(f"  [Rate Limited] Hourly limit hit. Pausing for 300s...")
                time.sleep(300)
                continue
            else:
                return "DAILY_LIMIT"
                
        if response.status_code != 200:
            print(f"  [Error] HTTP {response.status_code} at {lat}, {lon}")
            return "ERROR"
            
        data = response.json()
        
        df_daily = pd.DataFrame(data.get('daily', {}))
        if df_daily.empty:
            return "ERROR"
            
        df_hourly = pd.DataFrame(data.get('hourly', {}))
        if not df_hourly.empty:
            df_hourly['date'] = df_hourly['time'].str[:10]
            df_hourly['wind_speed_100m_cubed'] = df_hourly['wind_speed_100m'] ** 3
            
            daily_agg = df_hourly.groupby('date').agg({
                'wind_speed_100m': 'mean',
                'wind_speed_100m_cubed': 'mean',
                'snow_depth': 'mean' 
            }).reset_index()
            
            df_daily = pd.merge(df_daily, daily_agg, left_on='time', right_on='date', how='left')
            df_daily.drop(columns=['date'], inplace=True)
        
        # Output format is now purely geographic and temporal
        df_daily.insert(0, 'Lat_Rounded', lat)
        df_daily.insert(1, 'Lon_Rounded', lon)
        
        header = not os.path.exists(OUTPUT_FILE)
        df_daily.to_csv(OUTPUT_FILE, mode='a', index=False, header=header)
        
        return "SUCCESS"

def main():
    coords_df = load_coordinates()
    completed = get_completed_coordinates()
    
    total_requests = len(coords_df)
    requests_processed = len(completed)
    
    print("\n--- Starting Fetch Process (Pure Geographic Grid) ---")
    print(f"Found {total_requests} unique (Year -> Coordinate) physical targets.")
    print(f"Progress loaded: {requests_processed} items already secured in CSV.")
    
    for year in TARGET_YEARS:
        year_data = coords_df[coords_df['Year'] == year]
        if year_data.empty: continue
        
        print(f"\n=========================================")
        print(f" FETCHING METEOROLOGY FOR {year}")
        print(f"=========================================")
        
        remaining = [
            (int(r['Year']), round(r['Latitude_Round'], 2), round(r['Longitude_Round'], 2)) 
            for _, r in year_data.iterrows() 
            if (int(r['Year']), round(r['Latitude_Round'], 2), round(r['Longitude_Round'], 2)) not in completed
        ]
        
        if not remaining:
            continue
            
        print(f"Processing {len(remaining)} unique physical coordinates...")
        
        for index, row in year_data.iterrows():
            coord_tuple = (int(row['Year']), round(row['Latitude_Round'], 2), round(row['Longitude_Round'], 2))
            
            if coord_tuple in completed:
                continue
                
            status = fetch_and_aggregate_weather(row)
            
            if status == "DAILY_LIMIT":
                print("\n" + "="*60)
                print("🛑 DAILY API LIMIT REACHED (HTTP 429)!")
                print(f"Progress safely secured in '{OUTPUT_FILE}'.")
                print("Restart this script tomorrow. It will resume exactly where it left off.")
                print("="*60)
                sys.exit(0)
                
            elif status == "SUCCESS":
                requests_processed += 1
                print(f"  -> {coord_tuple[1]}, {coord_tuple[2]} Saved. ({requests_processed}/{total_requests} Total)")
                time.sleep(1.5) 
            else:
                print(f"  -> {coord_tuple[1]}, {coord_tuple[2]} Failed. Skipping.")
                time.sleep(2.0)
                    
    print(f"\nAll operations finished! Dataset is complete in {OUTPUT_FILE}.")

if __name__ == "__main__":
    main()

Weather Capacity-Weighting

This program merges weather data with the spatial grid; it multiplies weather variables by local technology-specific capacity, sums them by region and divides by the total regional capacity, this happens for each year.
Initially, variables were weighted against select technologies (shortwave radition only to solar) but this was changed; all weather variables are now weighted against every technology independently, which enables cross-technology market dynamics.
   - Output: Regional_Weighted_Weather_2023_2025.csv
   - Format: [Region, Date, {Tech}_{Weather_Variable}...] (34 columns covering all cross-technology weather profiles)

In [ ]:
#AI assisted code.
warnings.filterwarnings('ignore')

def build_regional_weather():
    print("--- Starting Regional Weather Weighting ---")
    
    # 1. Load the data 
    clusters = pd.read_csv("European_Renewable_Clusters.csv")
    weather = pd.read_csv("European_Weather_Data_2023_2025.csv")
    
    # 2. Extract Year from weather to precisely match the clustered time-travel logic
    weather['Date'] = weather['time'].str[:10]
    weather['Year'] = weather['Date'].str[:4].astype(int)
    
    # 3. Pivot Clusters by Technology
    pivot_cols = ['Region', 'Year', 'Latitude_Round', 'Longitude_Round']
    cluster_pivot = clusters.groupby(pivot_cols + ['Technology'])['Total_Capacity'].sum().unstack(fill_value=0).reset_index()
    
    techs = [c for c in ['Wind', 'Solar', 'Hydro', 'Bioenergy'] if c in cluster_pivot.columns]
    cluster_pivot['Total_Capacity'] = cluster_pivot[techs].sum(axis=1)
    
    # 4. Merge Weather with Cluster Weights
    merged = pd.merge(
        weather, 
        cluster_pivot, 
        left_on=['Lat_Rounded', 'Lon_Rounded', 'Year'], 
        right_on=['Latitude_Round', 'Longitude_Round', 'Year'],
        how='inner'
    )
    
    # 5. Apply the Technology-Specific Mathematical Weights (Matrix Approach)
    weather_vars = [
        'shortwave_radiation_sum', 'temperature_2m_max', 'wind_gusts_10m_max', 
        'precipitation_sum', 'apparent_temperature_min', 'wind_speed_100m', 
        'wind_speed_100m_cubed', 'snow_depth'
    ]
    
    # Create the weighted numerators
    for tech in techs:
        for w_var in weather_vars:
            merged[f'w_{tech}_{w_var}'] = merged[w_var] * merged[tech]
            
    # 6. Sum the weighted numerators by Region and Date
    agg_dict = {t: 'sum' for t in techs}
    weight_cols = [c for c in merged.columns if c.startswith('w_')]
    for wc in weight_cols:
        agg_dict[wc] = 'sum'
        
    regional = merged.groupby(['Region', 'Date']).agg(agg_dict).reset_index()
    
    # 7. Divide by the exact technological denominator
    for tech in techs:
        safe_denominator = regional[tech].replace(0, pd.NA)
        for w_var in weather_vars:
            # Re-capitalize the final output names for the graphing script
            clean_var = w_var.title().replace('_', '')
            # We keep the underscores to match the visualizer's segmented control format exactly
            col_name = f"{tech}_{w_var.title()}" 
            regional[col_name] = (regional[f'w_{tech}_{w_var}'] / safe_denominator).round(2)

    # 8. Cleanup and Export
    final_cols = ['Region', 'Date'] + [c for c in regional.columns if '_' in c and c.startswith(tuple(techs))]
    final_df = regional[final_cols]
    
    final_df.to_csv("Regional_Weighted_Weather_2023_2025.csv", index=False)
    print("✓ Successfully calculated technology-specific weighted weather.")
    
if __name__ == "__main__":
    build_regional_weather()

Energy Generation and Capacity Fetching

This program fetches energy generation and annual capacity data from each country from Energy-Charts API. 
It also aggregates sub-categories, handles NaNs via preservation to avoid skewed data, and fills missing technologies with 0.0.
Each country's data is output into its own file for easier manual analysis.
   - Output A (Dir): Renewable_Generation_By_Country/ [Time (Local), Bioenergy, Hydro, Wind, Solar]
   - Output B (Dir): Installed_Capacity_By_Country/ [Year, Bioenergy, Hydro, Wind, Solar]

In [ ]:
#AI assisted code.
def main():
    countries = {
        'Denmark': 'dk', 'Sweden': 'se', 'Finland': 'fi', 
        'Lithuania': 'lt', 'Ireland': 'ie', 'Estonia': 'ee', 'Latvia': 'lv', 
        'Norway': 'no', 'Portugal': 'pt', 'Italy': 'it', 'Spain': 'es', 
        'Serbia': 'rs', 'Croatia': 'hr', 'Bosnia and Herzegovina': 'ba', 
        'Greece': 'gr', 'Montenegro': 'me', 'Slovenia': 'si', 'North Macedonia': 'mk'
    }

    years = [2023, 2024, 2025]
    
    gen_output_dir = "Renewable_Generation_By_Country"
    cap_output_dir = "Installed_Capacity_By_Country"
    
    os.makedirs(gen_output_dir, exist_ok=True)
    os.makedirs(cap_output_dir, exist_ok=True)

    # TAXONOMY MAPPING
    gen_target_techs = ['Biomass', 'Waste', 'Hydro Run-of-River', 'Hydro water reservoir', 'Hydro pumped storage', 'Solar', 'Wind onshore', 'Wind offshore']
    cap_target_techs = ['Biomass', 'Waste', 'Hydro', 'Hydro Run-of-River', 'Hydro water reservoir', 'Hydro pumped storage', 'Solar DC', 'Solar AC', 'Wind onshore', 'Wind offshore']

    gen_agg_map = {
        'Bioenergy': ['Biomass', 'Waste'],
        'Hydro': ['Hydro Run-of-River', 'Hydro water reservoir', 'Hydro pumped storage'],
        'Wind': ['Wind onshore', 'Wind offshore'],
        'Solar': ['Solar']
    }
    
    cap_agg_map = {
        'Bioenergy': ['Biomass', 'Waste'],
        'Hydro': ['Hydro', 'Hydro Run-of-River', 'Hydro water reservoir', 'Hydro pumped storage'],
        'Wind': ['Wind onshore', 'Wind offshore'],
        'Solar': ['Solar DC', 'Solar AC'] 
    }
    
    FINAL_TECHS = ['Bioenergy', 'Hydro', 'Wind', 'Solar']

    def aggregate_technologies(df, is_capacity=False):
        """Fuses subcategories, normalizes the column structure, and handles missing data."""
        time_col = 'Year' if is_capacity else 'Time (Local)'
        mapping = cap_agg_map if is_capacity else gen_agg_map
        
        # 1. Fuse subcategories (e.g., Biomass + Waste -> Bioenergy)
        for final_tech, sub_techs in mapping.items():
            if final_tech == 'Solar': continue
            existing_cols = [col for col in sub_techs if col in df.columns]
            if existing_cols:
                df[final_tech] = df[existing_cols].sum(axis=1, min_count=1)
                
        # 2. Process Solar explicitly to prevent AC/DC double counting
        if 'Solar DC' in df.columns and 'Solar AC' in df.columns:
            df['Solar'] = df['Solar DC'].fillna(df['Solar AC'])
        elif 'Solar DC' in df.columns:
            df['Solar'] = df['Solar DC']
        elif 'Solar AC' in df.columns:
            df['Solar'] = df['Solar AC']
            
        # 3. Unit Conversion for Capacity
        if is_capacity:
            for tech in FINAL_TECHS:
                if tech in df.columns:
                    df[tech] = df[tech] * 1000.0
                    
        # 4. NORMALIZATION: Force all 4 columns to exist
        for tech in FINAL_TECHS:
            if tech not in df.columns:
                df[tech] = 0.0 # Country lacks the infrastructure completely
                
        # 5. Missing Row & NaN Handling
        if is_capacity:
            # Force all years (2023, 2024, 2025) to exist in the dataframe
            all_years = pd.DataFrame({'Year': years})
            df = pd.merge(all_years, df, on='Year', how='left')
            # Fill missing capacity values (NaNs) with 0.0
            df[FINAL_TECHS] = df[FINAL_TECHS].fillna(0.0)
        else:
            # For generation, missing columns got 0.0 above. 
            # We DO NOT run fillna(0.0) on existing columns to preserve telemetry NaNs.
            pass

        # Return strictly the 5 requested columns in exact order
        return df[[time_col] + FINAL_TECHS]

    print("--- Starting Energy-Charts API Fetch (Hardened & Normalized) ---")
    
    for country, code in countries.items():
        print(f"\nProcessing {country} ({code.upper()})...")
        
        # 1. Fetch Installed Capacity
        cap_url = f"https://api.energy-charts.info/v2/installed_power?country={code}&time_step=yearly"
        try:
            cap_response = requests.get(cap_url)
            if cap_response.status_code == 200:
                payload = cap_response.json()
                data_points = payload.get('data', [])
                
                cap_rows = []
                if data_points:
                    series_info = payload.get('series', [])
                    id_to_name = {s['id']: s['name'] for s in series_info}
                    
                    for dp in data_points:
                        year_str = dp.get('timestamp', '')[:4]
                        if year_str.isdigit() and int(year_str) in years:
                            row = {'Year': int(year_str)}
                            for s_id, val in dp.get('values', {}).items():
                                name = id_to_name.get(s_id)
                                if name in cap_target_techs:
                                    row[name] = val
                            cap_rows.append(row)
                            
                df_cap = pd.DataFrame(cap_rows) if cap_rows else pd.DataFrame(columns=['Year'])
                
                # Normalize and save
                df_cap = aggregate_technologies(df_cap, is_capacity=True)
                file_path = os.path.join(cap_output_dir, f"{country}_Capacity_2023_2025.csv")
                df_cap.to_csv(file_path, index=False)
                print(f"  -> Saved {country} Capacity data (Normalized).")
                
            else:
                print(f"  [Error] Capacity fetch failed with HTTP {cap_response.status_code}")
        except Exception as e:
            print(f"  [Error] Capacity fetch exception: {str(e)}")
            
        time.sleep(1.5)
        
        # 2. Fetch Daily Generation
        country_yearly_dfs = []
        for year in years:
            url = f"https://api.energy-charts.info/v2/public_power?country={code}&start={year}-01-01&end={year}-12-31"
            
            max_retries = 5
            for attempt in range(max_retries):
                try:
                    response = requests.get(url)
                    
                    if response.status_code == 429:
                        wait_time = 15 * (attempt + 1)
                        print(f"  [Rate Limited] Pausing for {wait_time}s...")
                        time.sleep(wait_time)
                        continue
                        
                    if response.status_code != 200:
                        break 
                        
                    payload = response.json()
                    data_points = payload.get('data', [])
                    if not data_points:
                        break
                    
                    series_info = payload.get('series', [])
                    id_to_name = {s['id']: s['name'] for s in series_info}
                    
                    rows = []
                    for dp in data_points:
                        row = {'Time (Local)': dp.get('timestamp')}
                        for s_id, val in dp.get('values', {}).items():
                            name = id_to_name.get(s_id)
                            if name in gen_target_techs:
                                row[name] = val
                        rows.append(row)
                        
                    country_yearly_dfs.append(pd.DataFrame(rows))
                    print(f"  [Success] Generation {year} fetched.")
                    break 
                    
                except Exception as e:
                    time.sleep(5)
            time.sleep(1.5) 

        if country_yearly_dfs:
            df_country_all = pd.concat(country_yearly_dfs, ignore_index=True)
            df_country_all.sort_values('Time (Local)', inplace=True)
            
            # Normalize and save
            df_country_all = aggregate_technologies(df_country_all, is_capacity=False)
            file_path = os.path.join(gen_output_dir, f"{country}_Generation_2023_2025.csv")
            df_country_all.to_csv(file_path, index=False)
            print(f"  -> Saved {country} Generation data (Normalized).")
        else:
            print(f"  -> No generation data found for {country}.")

if __name__ == "__main__":
    main()

Generation Normalization 

This program identifies telemetry intervals (15, 30, or 60 min), and converts MW to MWh and aggregates them into daily totals.
It does not rely on the API response's interval value, since it sometimes ouputs null.
Intervals are calculated using UTC to account for DST.
To prevent skewed data, it discards days with fewer than 22 hours of valid telemetry.
Each country's data is output into its own file for easier manual analysis.
   - Output (Dir): Daily_Generation_MWh/ [Country, Date, Bioenergy, Hydro, Wind, Solar]

In [ ]:
#AI assisted code.
warnings.filterwarnings('ignore')

INPUT_DIR = "Renewable_Generation_By_Country"
OUTPUT_DIR = "Daily_Generation_MWh"
TARGET_TECHS = ['Bioenergy', 'Hydro', 'Wind', 'Solar']

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
    if not files:
        print(f"No CSV files found in '{INPUT_DIR}'.")
        return

    print("--- Starting Simplified Sub-Daily to Daily MWh Normalization ---")
    
    for file_path in files:
        filename = os.path.basename(file_path)
        country = filename.split('_')[0]
        
        df = pd.read_csv(file_path)
        if df.empty or len(df) < 2:
            continue
            
        # 1. Parse timestamps
        # Use UTC ONLY for math to bypass Daylight Saving Time overlaps/skips
        df['UTC'] = pd.to_datetime(df['Time (Local)'], utc=True)
        df = df.sort_values('UTC').reset_index(drop=True)
        
        # Extract the Year and Date strictly from the Local Time string
        df['Year'] = df['Time (Local)'].str[:4].astype(int)
        df['Date'] = df['Time (Local)'].str[:10]
        
        # 2. Determine the static interval per local year
        interval_map = {}
        for year, group in df.groupby('Year'):
            if len(group) > 1:
                # Mode returns the most common gap. Extract total seconds and convert to hours.
                mode_seconds = group['UTC'].diff().dt.total_seconds().mode()[0]
                interval_map[year] = mode_seconds / 3600.0
            else:
                interval_map[year] = 1.0 # Fallback 
                
        # 3. Map the calculated interval back to every row based on its local year
        df['Interval_Hours'] = df['Year'].map(interval_map)
        
        # 4. Convert MW to MWh
        for tech in TARGET_TECHS:
            if tech in df.columns:
                df[tech] = df[tech] * df['Interval_Hours']
                
        # 5. Group into Daily totals and enforce the 22-Hour Safety Threshold
        agg_funcs = {tech: 'sum' for tech in TARGET_TECHS if tech in df.columns}
        agg_funcs['Interval_Hours'] = 'sum'
        
        daily_df = df.groupby('Date').agg(agg_funcs).reset_index()
        
        # If a day recorded less than 22 hours of local data, discard it
        invalid_days = daily_df['Interval_Hours'] < 22.0
        for tech in TARGET_TECHS:
            if tech in daily_df.columns:
                daily_df.loc[invalid_days, tech] = pd.NA
                
        # 6. Final Cleanup
        daily_df.insert(0, 'Country', country)
        daily_df = daily_df.drop(columns=['Interval_Hours'])
        
        out_path = os.path.join(OUTPUT_DIR, filename)
        daily_df.to_csv(out_path, index=False)
        
        intervals_detected = [f"{year}: {int(hrs * 60)}m" for year, hrs in interval_map.items()]
        print(f"✓ Processed {country} | Intervals -> {', '.join(intervals_detected)}")
        
    print(f"\nAll files correctly aggregated and stored in '{OUTPUT_DIR}'.")

if __name__ == "__main__":
    main()

Capacity Baseline Estimation

From the manually edited GEM files, it determines each country's minimum national capacity baseline per technology, per year. 
It assumes 0.0 for missing technologies.
   - Output: GEM_Estimated_Capacities_2023_2025.csv
   - Format: [Country, Year, Bioenergy, Hydro, Solar, Wind]

In [ ]:
#AI assisted code.
def generate_capacity_estimates():
    # Map your 4 target indices to the respective simplified GEM files
    tracker_files = {
        'Hydro': 'European_Hydro_Data.xlsx',
        'Wind': 'European_Wind_Data.xlsx',
        'Solar': 'European_Solar_Data.xlsx',
        'Bioenergy': 'European_Bioenergy_Data.xlsx'
    }
    
    target_years = [2023, 2024, 2025]
    all_results = []
    
    for tech, filepath in tracker_files.items():
        print(f"Aggregating {tech} data from {filepath}...")
        df = pd.read_excel(filepath)
        
        for year in target_years:
            # 1. START YEAR LOGIC
            # If a plant was commissioned in 2023, it generated power in 2023, so we count it.
            # If the Start Year is NaN, it is an active/retired plant with an unknown historical 
            # build date. We safely assume it was built before our 2023-2025 window.
            started = df['Start Year'].isna() | (df['Start Year'] <= year)
            
            # 2. RETIRED YEAR LOGIC
            # If a plant retired in 2023, it still contributed to the grid for part of that year, 
            # so we keep it in the 2023 pool but mathematically drop it in 2024.
            # If the Retired Year is NaN, the plant is still actively operating.
            not_retired = df['Retired Year'].isna() | (df['Retired Year'] >= year)
            
            # Filter the active fleet for this specific year
            active_fleet = df[started & not_retired]
            
            # Sum the capacity per country
            annual_capacity = active_fleet.groupby('Country')['Capacity (MW)'].sum().reset_index()
            annual_capacity['Year'] = year
            annual_capacity['Technology'] = tech
            
            all_results.append(annual_capacity)
            
    # Combine all technologies and years into a single dataframe
    master_df = pd.concat(all_results, ignore_index=True)
    
    # Pivot the data so each technology gets its own column, filling missing techs with 0.0
    final_df = master_df.pivot_table(
        index=['Country', 'Year'], 
        columns='Technology', 
        values='Capacity (MW)', 
        fill_value=0.0
    ).reset_index()
    
    output_name = 'GEM_Estimated_Capacities_2023_2025.csv'
    final_df.to_csv(output_name, index=False)
    print(f"\n✓ Success! Master fallback file saved to '{output_name}'")

if __name__ == "__main__":
    generate_capacity_estimates()

Capacity Validation

This program compares official EnergyCharts capacity against GEM estimations and retains the maximum value to correct underreported national data.
   - Output (Dir): Final_Validated_Capacity/ [Year, Bioenergy, Hydro, Wind, Solar]

In [ ]:
#AI assisted code.
INPUT_DIR = "Installed_Capacity_By_Country"
OUTPUT_DIR = "Final_Validated_Capacity"
GEM_FILE = "GEM_Estimated_Capacities_2023_2025.csv"
TARGET_TECHS = ['Bioenergy', 'Hydro', 'Wind', 'Solar']

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if not os.path.exists(GEM_FILE):
        print(f"[Error] Missing '{GEM_FILE}'.")
        return
        
    gem_df = pd.read_csv(GEM_FILE)
    files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
    
    print("--- Starting Capacity Validation (EC vs GEM) ---")
    
    for file_path in files:
        filename = os.path.basename(file_path)
        country = filename.split('_')[0]
        
        cap_df = pd.read_csv(file_path)
        gem_country_df = gem_df[gem_df['Country'] == country]
        
        if not gem_country_df.empty:
            for year in [2023, 2024, 2025]:
                cap_mask = cap_df['Year'] == year
                gem_mask = gem_country_df['Year'] == year
                
                if cap_mask.any() and gem_mask.any():
                    for tech in TARGET_TECHS:
                        ec_val = cap_df.loc[cap_mask, tech].values[0]
                        gem_val = gem_country_df.loc[gem_mask, tech].values[0]
                        # Directly overwrite with whichever physical capacity is larger
                        cap_df.loc[cap_mask, tech] = max(ec_val, gem_val)
                        
        out_path = os.path.join(OUTPUT_DIR, filename)
        cap_df.to_csv(out_path, index=False)
        print(f"✓ Validated {country}")
        
    print(f"\nAll files saved to '{OUTPUT_DIR}'.")

if __name__ == "__main__":
    main()

Master Aggregation
This programs merges all country-level files from the output directories into the final project master files.
- Output A: European_Daily_Generation_2023_2025.csv [Region, Country, Date, Bioenergy, Hydro, Wind, Solar]
- Output B: European_Validated_Capacity_2023_2025.csv [Region, Country, Year, Bioenergy, Hydro, Wind, Solar]

In [ ]:
#AI assisted code.
warnings.filterwarnings('ignore')

GEN_DIR = "Daily_Generation_MWh"
CAP_DIR = "Final_Validated_Capacity"

OUT_GEN_FILE = "European_Daily_Generation_2023_2025.csv"
OUT_CAP_FILE = "European_Validated_Capacity_2023_2025.csv"

NORTHERN_EUROPE = ['Denmark', 'Sweden', 'Finland', 'Lithuania', 'Ireland', 'Estonia', 'Latvia', 'Norway']
SOUTHERN_EUROPE = ['Portugal', 'Italy', 'Spain', 'Serbia', 'Croatia', 'Bosnia and Herzegovina', 'Greece', 'Montenegro', 'Slovenia', 'North Macedonia']

TECHS = ['Bioenergy', 'Hydro', 'Wind', 'Solar']

def get_region(country):
    if country in NORTHERN_EUROPE: return 'Northern Europe'
    if country in SOUTHERN_EUROPE: return 'Southern Europe'
    return 'Unknown'

def aggregate_capacity():
    print(f"--- Aggregating Capacity Data from '{CAP_DIR}/' ---")
    cap_list = []
    
    if not os.path.exists(CAP_DIR):
        print(f"[!] Error: Directory '{CAP_DIR}' not found.")
        return

    for file in os.listdir(CAP_DIR):
        if file.endswith(".csv"):
            # Extract the strict country name from the filename (e.g., "Spain_Capacity_2023_2025.csv" -> "Spain")
            country = file.split("_Capacity")[0]
            
            df = pd.read_csv(os.path.join(CAP_DIR, file))
            
            # Enforce strict column standardization
            df['Country'] = country
            df['Region'] = get_region(country)
            
            # Reorder columns for readability
            cols = ['Region', 'Country', 'Year'] + [t for t in TECHS if t in df.columns]
            cap_list.append(df[cols])
            
    if cap_list:
        all_cap = pd.concat(cap_list, ignore_index=True)
        # Sort geographically and chronologically
        all_cap = all_cap.sort_values(by=['Region', 'Country', 'Year'])
        all_cap.to_csv(OUT_CAP_FILE, index=False)
        print(f"✓ Saved {len(all_cap)} rows to {OUT_CAP_FILE}")
    else:
        print("[!] No capacity CSVs found.")

def aggregate_generation():
    print(f"\n--- Aggregating Generation Data from '{GEN_DIR}/' ---")
    gen_list = []
    
    if not os.path.exists(GEN_DIR):
        print(f"[!] Error: Directory '{GEN_DIR}' not found.")
        return

    for file in os.listdir(GEN_DIR):
        if file.endswith(".csv"):
            country = file.split("_Generation")[0]
            df = pd.read_csv(os.path.join(GEN_DIR, file))

            # 1. Standardize Timestamps
            # If a country (like Finland) uses hourly 'Time (Local)', convert it to YYYY-MM-DD
            if 'Time (Local)' in df.columns:
                df['Date'] = pd.to_datetime(df['Time (Local)'], utc=True).dt.strftime('%Y-%m-%d')
                # Sum the 24 hourly rows into 1 daily row for the specific technologies
                df = df.groupby('Date')[TECHS].sum().reset_index()
            elif 'Date' in df.columns:
                # Ensure existing dates are safely formatted strings
                df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

            # 2. Append Geographic Metadata
            df['Country'] = country
            df['Region'] = get_region(country)
            
            # 3. Reorder and secure columns
            cols = ['Region', 'Country', 'Date'] + [t for t in TECHS if t in df.columns]
            gen_list.append(df[cols])
            
    if gen_list:
        all_gen = pd.concat(gen_list, ignore_index=True)
        all_gen = all_gen.sort_values(by=['Region', 'Country', 'Date'])
        all_gen.to_csv(OUT_GEN_FILE, index=False)
        print(f"✓ Saved {len(all_gen)} rows to {OUT_GEN_FILE}")
    else:
        print("[!] No generation CSVs found.")

if __name__ == "__main__":
    aggregate_capacity()
    aggregate_generation()
    print("\nAggregation complete. You can now use the master files for visualization.")

This marks the end of the pipeline. The resulting files can be found in the corresponding data folder of the project, and how they are utilized is documented in the streamlit program files.